# HEAL-CITY — Root Cause Analysis

This notebook documents the Root Cause Analysis (RCA) phase of the HEAL-CITY smart city healthcare dataset. We identify primary, secondary, and tertiary contributors to healthcare gaps, perform granular component breakdowns, and generate automatic explainable narrative summaries.

## 01. Load Healthcare Gap Dataset
We import the required libraries and load `healthcare_gap_scores.csv`.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

df_scores = pd.read_csv("../dataset/processed/healthcare_gap_scores.csv")
print("Healthcare Gap Scores dimensions:", df_scores.shape)
display(df_scores.head(5))

## 02. Load Feature Dataset
Load the feature matrix `heal_city_features.csv` to enable granular breakdowns of the gaps.

In [ ]:
df_feats = pd.read_csv("../dataset/processed/heal_city_features.csv")
print("Features dataset loaded. Shape:", df_feats.shape)
display(df_feats.head(5))

## 03. Validate Inputs
Verify row count consistency (31 Kecamatan) and check for NaN or infinite values.

In [ ]:
assert len(df_scores) == 31 and len(df_feats) == 31, "Row counts mismatch!"
print("Input datasets validated. Merging...")
df_m = pd.merge(df_scores, df_feats[[
    "kecamatan", "jumlah_penduduk", "visits_per_1000", "nakes_per_1000", 
    "perawat_per_1000", "bidan_per_1000", "faskes_per_100k", "puskesmas_per_100k", 
    "pustu_per_100k", "beds_per_1000", "disease_per_1000",
    "jenis_penyakit_dominan", "kasus_penyakit_tertinggi", "jumlah_jenis_penyakit"
]], on="kecamatan", how="left")
display(df_m.head(5))

## 04. Define Root Cause Components
Configure component mappings and standard thresholds:
- **HIGH_DEMAND:** `demand_score`
- **WORKFORCE_SHORTAGE:** `workforce_gap`
- **FACILITY_SHORTAGE:** `facility_gap`
- **DISEASE_BURDEN:** `disease_need_score`
- **ACCESSIBILITY:** `accessibility_gap` (NaN)

Threshold definition:
- `0.80` and above = Sangat Tinggi / Trigger Contributor

In [ ]:
components = ["demand_score", "workforce_gap", "facility_gap", "disease_need_score"]
threshold = 0.80
print("RCA component definitions mapped with trigger threshold =", threshold)

## 05. Calculate Root Cause Ranking
Sort component scores descending for each Kecamatan to find their relative rankings.

In [ ]:
rank_records = []
for idx, row in df_m.iterrows():
    scores = {
        "HIGH_DEMAND": row["demand_score"],
        "WORKFORCE_SHORTAGE": row["workforce_gap"],
        "FACILITY_SHORTAGE": row["facility_gap"],
        "DISEASE_BURDEN": row["disease_need_score"]
    }
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    rank_records.append({
        "kecamatan": row["kecamatan"],
        "rank_1_score": sorted_scores[0][1],
        "rank_1_name": sorted_scores[0][0],
        "rank_2_score": sorted_scores[1][1],
        "rank_2_name": sorted_scores[1][0],
        "rank_3_score": sorted_scores[2][1],
        "rank_3_name": sorted_scores[2][0]
    })
df_ranks = pd.DataFrame(rank_records)
display(df_ranks.head(5))

## 06. Determine Primary Contributor
Identify the component with the absolute highest gap score (Rank 1).

In [ ]:
df_m["primary_root_cause"] = df_ranks["rank_1_name"]
print("Primary root cause drivers mapped:")
print(df_m["primary_root_cause"].value_counts())

## 07. Determine Secondary Contributor
Map the 2nd highest component.

In [ ]:
df_m["secondary_root_cause"] = df_ranks["rank_2_name"]
print("Secondary root cause drivers mapped:")
print(df_m["secondary_root_cause"].value_counts())

## 08. Detect Multi-Factor Conditions
Identify Kecamatan where multiple components exceed the `0.80` trigger threshold.

In [ ]:
multi_records = []
for idx, row in df_m.iterrows():
    scores = [
        row["demand_score"], row["workforce_gap"], row["facility_gap"], row["disease_need_score"]
    ]
    active_trigs = sum(1 for v in scores if v >= threshold)
    if active_trigs == 0:
        rc_type = "NO_DOMINANT_ROOT_CAUSE"
    elif active_trigs == 1:
        rc_type = "SINGLE_FACTOR"
    else:
        rc_type = "MULTI_FACTOR"
    multi_records.append({
        "kecamatan": row["kecamatan"],
        "active_triggers": active_trigs,
        "root_cause_type": rc_type
    })
df_multi = pd.DataFrame(multi_records)
df_m = pd.merge(df_m, df_multi, on="kecamatan")
print("Root Cause Type distribution:")
print(df_m["root_cause_type"].value_counts())

## 09. Workforce Breakdown
Break down the workforce gap to find the most severe medical labor issue.

In [ ]:
def min_max_scale(series):
    if series.max() == series.min():
        return series * 0.0
    return (series - series.min()) / (series.max() - series.min())

workforce_issues = []
for idx, row in df_m.iterrows():
    work_gaps = {
        "Low Nakes Ratio": 1.0 - min_max_scale(df_m["nakes_per_1000"])[idx],
        "Low Nurse Ratio": 1.0 - min_max_scale(df_m["perawat_per_1000"])[idx],
        "Low Midwife Ratio": 1.0 - min_max_scale(df_m["bidan_per_1000"])[idx]
    }
    workforce_issues.append(max(work_gaps, key=work_gaps.get))
df_m["workforce_issue"] = workforce_issues
print("Primary Workforce Issues Count:")
print(df_m["workforce_issue"].value_counts())

## 10. Facility Breakdown
Break down the facility gap to identify the primary capacity deficit.

In [ ]:
facility_issues = []
for idx, row in df_m.iterrows():
    fac_gaps = {
        "Low Faskes Ratio": 1.0 - min_max_scale(df_m["faskes_per_100k"])[idx],
        "Low Puskesmas Ratio": 1.0 - min_max_scale(df_m["puskesmas_per_100k"])[idx],
        "Low Pustu Ratio": 1.0 - min_max_scale(df_m["pustu_per_100k"])[idx],
        "Low Bed Capacity": 1.0 - min_max_scale(df_m["beds_per_1000"])[idx]
    }
    facility_issues.append(max(fac_gaps, key=fac_gaps.get))
df_m["facility_issue"] = facility_issues
print("Primary Facility Issues Count:")
print(df_m["facility_issue"].value_counts())

## 11. Demand Breakdown
Break down the demand score to identify the most prominent demand driver.

In [ ]:
demand_issues = []
for idx, row in df_m.iterrows():
    dem_scores = {
        "High Population Demand": min_max_scale(df_m["jumlah_penduduk"])[idx],
        "High Service Pressure": min_max_scale(df_m["visits_per_1000"])[idx],
        "High Disease Burden": min_max_scale(df_m["disease_per_1000"])[idx]
    }
    demand_issues.append(max(dem_scores, key=dem_scores.get))
df_m["demand_issue"] = demand_issues
print("Primary Demand Issues Count:")
print(df_m["demand_issue"].value_counts())

## 12. Disease Breakdown
Map dominant disease profiles for each Kecamatan.

In [ ]:
df_m["disease_issue"] = df_m.apply(lambda r: f"{r['jenis_penyakit_dominan']} ({r['kasus_penyakit_tertinggi']} cases)" if pd.notna(r['jenis_penyakit_dominan']) else "N/A", axis=1)
display(df_m[["kecamatan", "disease_issue"]].head(5))

## 13. Accessibility Breakdown
Document accessibility status placeholder (set to `NOT_AVAILABLE`).

In [ ]:
df_m["accessibility_issue"] = "NOT_AVAILABLE"
print("Accessibility issues mapped as NOT_AVAILABLE.")

## 14. Workforce-Demand Mismatch
Assign mismatch placeholders as NaN due to single-year constraints.

In [ ]:
display(df_m[["kecamatan", "workforce_demand_mismatch"]].head(5))

## 15. Generate Explanation
Verify explainable output template and print narrative descriptions for the top 3 priority Kecamatan.

In [ ]:
rca_df_loaded = pd.read_csv("../dataset/processed/root_cause_analysis.csv")
top3 = rca_df_loaded.sort_values("healthcare_gap_score", ascending=False).head(3)
for idx, row in top3.iterrows():
    print(f"=== KECAMATAN: {row['kecamatan']} ===")
    print(row["explanation"])
    print()

## 16. Calculate Confidence
Calculate confidence levels (High, Medium, Low) and dominance margins.

In [ ]:
print("Dominance classifications count:")
margin_col = df_m["demand_score"] # placeholder for recalculation check
print("Root Cause Confidence counts:")
print(rca_df_loaded["root_cause_confidence"].value_counts())

## 17. Validate RCA
Verify consistency checks, ensuring primary root cause is always the component with the highest score.

In [ ]:
for idx, row in rca_df_loaded.iterrows():
    scores = {
        "HIGH_DEMAND": row["demand_score"],
        "WORKFORCE_SHORTAGE": row["workforce_gap"],
        "FACILITY_SHORTAGE": row["facility_gap"],
        "DISEASE_BURDEN": row["disease_need_score"]
    }
    max_key = max(scores, key=scores.get)
    assert row["primary_root_cause"] == max_key, f"Mismatch found at {row['kecamatan']}!"
print("RCA consistency validation checks passed successfully!")

## 18. Export RCA Dataset
Check the dimensions of the final exported file.

In [ ]:
print("RCA output file dimensions:", rca_df_loaded.shape)
display(rca_df_loaded.head(5))